In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
import os
# Store Hugging Face models/cache on D: drive
os.environ["HF_HOME"] = "D:/huggingface_cache"

In [4]:
embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4535.53it/s]


In [5]:
from langchain_chroma import Chroma

In [6]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [7]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [8]:
vector_store = Chroma(
    embedding_function = embedding,
    persist_directory = 'my_chroma_db',
    collection_name = 'sample'
)

In [9]:
# add documents
vector_store.add_documents(docs)

['12e42ac7-438b-456a-957e-40e69281e9af',
 '8963c71f-3794-4346-83b6-a857cf876d92',
 'ff9efe71-5cf3-4009-8d97-b804fc32a356',
 '078b9d2a-e7ab-4f42-b885-12651ab1f8d4',
 'ef870f47-bed8-466d-af5d-61ccfa8f299d']

In [10]:
# view documents
vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

{'ids': ['12e42ac7-438b-456a-957e-40e69281e9af',
  '8963c71f-3794-4346-83b6-a857cf876d92',
  'ff9efe71-5cf3-4009-8d97-b804fc32a356',
  '078b9d2a-e7ab-4f42-b885-12651ab1f8d4',
  'ef870f47-bed8-466d-af5d-61ccfa8f299d'],
 'embeddings': array([[ 0.00994724,  0.06914337, -0.05147111, ..., -0.03543338,
          0.01284805,  0.0124829 ],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280613,  0.02737714],
        [-0.10265914,  0.02650809,  0.02271499, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123391, -0.02468549, -0.04494374, ..., -0.10995812,
          0.00572557,  0.09915379],
        [ 0.01873984,  0.04382846, -0.04304255, ..., -0.07801618,
         -0.0784068 , -0.00304188]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [11]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='078b9d2a-e7ab-4f42-b885-12651ab1f8d4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='8963c71f-3794-4346-83b6-a857cf876d92', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [12]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='078b9d2a-e7ab-4f42-b885-12651ab1f8d4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104),
 (Document(id='8963c71f-3794-4346-83b6-a857cf876d92', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1493449211120605)]

In [13]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='ff9efe71-5cf3-4009-8d97-b804fc32a356', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(id='ef870f47-bed8-466d-af5d-61ccfa8f299d', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [14]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['12e42ac7-438b-456a-957e-40e69281e9af',
  '8963c71f-3794-4346-83b6-a857cf876d92',
  'ff9efe71-5cf3-4009-8d97-b804fc32a356',
  '078b9d2a-e7ab-4f42-b885-12651ab1f8d4',
  'ef870f47-bed8-466d-af5d-61ccfa8f299d'],
 'embeddings': array([[ 0.00994724,  0.06914337, -0.05147111, ..., -0.03543338,
          0.01284805,  0.0124829 ],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280613,  0.02737714],
        [-0.10265914,  0.02650809,  0.02271499, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123391, -0.02468549, -0.04494374, ..., -0.10995812,
          0.00572557,  0.09915379],
        [ 0.01873984,  0.04382846, -0.04304255, ..., -0.07801618,
         -0.0784068 , -0.00304188]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [16]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['12e42ac7-438b-456a-957e-40e69281e9af',
  '8963c71f-3794-4346-83b6-a857cf876d92',
  'ff9efe71-5cf3-4009-8d97-b804fc32a356',
  '078b9d2a-e7ab-4f42-b885-12651ab1f8d4',
  'ef870f47-bed8-466d-af5d-61ccfa8f299d'],
 'embeddings': array([[ 0.00994724,  0.06914337, -0.05147111, ..., -0.03543338,
          0.01284805,  0.0124829 ],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280613,  0.02737714],
        [-0.10265914,  0.02650809,  0.02271499, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123391, -0.02468549, -0.04494374, ..., -0.10995812,
          0.00572557,  0.09915379],
        [ 0.01873984,  0.04382846, -0.04304255, ..., -0.07801618,
         -0.0784068 , -0.00304188]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo